# **Pearl-AQI-Engine:** Data Ingestion & Feature Engineering
**Objective:** Fetch 90 days of historical weather and air quality data, structure it, and engineer predictive temporal and momentum features.
**Target Location:** Islamabad, Pakistan (Lat: 33.68, Lon: 73.05)

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta


pd.set_option('display.max_columns', None) #display settings to view dataframes


### **1. Data Ingestion**
Using the OpenMeteo Air Quality and Historical APIs. We bypass heavy client libraries to parse the raw JSON directly, giving us tighter control over the extraction process.

In [4]:
def fetch_historical_aqi(lat, lon, days=90):

    end_date = datetime.utcnow().date()#calculate rolling window for backfill
    start_date = end_date - timedelta(days=days)

    url = "https://air-quality-api.open-meteo.com/v1/air-quality"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "hourly": ["european_aqi", "pm10", "pm2_5", "nitrogen_dioxide"]
    }

    response = requests.get(url, params=params)
    response.raise_for_status() #fail fast if the API rejects the request

    data = response.json()

    df = pd.DataFrame(data['hourly'])
    df['time'] = pd.to_datetime(df['time'])

    return df


df_raw = fetch_historical_aqi(33.6844, 73.0479) #coordinates for isb
df_raw.head()

/tmp/ipykernel_8764/2788599426.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_date = datetime.utcnow().date()#calculate rolling window for backfill


,time,european_aqi,pm10,pm2_5,nitrogen_dioxide
0,2026-02-06 00:00:00,73,23.7,21.9,11.1
1,2026-02-06 01:00:00,73,22.8,21.0,21.1
2,2026-02-06 02:00:00,73,26.7,24.7,34.5
3,2026-02-06 03:00:00,73,34.9,33.2,42.2
4,2026-02-06 04:00:00,73,41.1,39.3,38.9


### **2. Feature Engineering**
Raw API numbers aren't sufficient. To help the model understand pollution life-cycles, we need to extract the temporal context (when is it happening?) and calculate the `aqi_momentum` (is it accumulating or dispersing?).

In [6]:
def engineer_features(df):
    df_engineered = df.copy()


    df_engineered = df_engineered.dropna().reset_index(drop=True) #drop_nulls

    df_engineered['hour'] = df_engineered['time'].dt.hour
    df_engineered['day_of_week'] = df_engineered['time'].dt.dayofweek
    df_engineered['month'] = df_engineered['time'].dt.month


    df_engineered['aqi_momentum'] = df_engineered['european_aqi'].diff()

    df_engineered['aqi_momentum'] = df_engineered['aqi_momentum'].fillna(0)#first row have a NaN momentum because there is no previous hour to subtract.


    return df_engineered

df_features = engineer_features(df_raw)

df_features[['time', 'european_aqi', 'aqi_momentum', 'hour']].head(10)

,time,european_aqi,aqi_momentum,hour
0,2026-02-06 00:00:00,73,0.0,0
1,2026-02-06 01:00:00,73,0.0,1
2,2026-02-06 02:00:00,73,0.0,2
3,2026-02-06 03:00:00,73,0.0,3
4,2026-02-06 04:00:00,73,0.0,4
5,2026-02-06 05:00:00,73,0.0,5
6,2026-02-06 06:00:00,73,0.0,6
7,2026-02-06 07:00:00,73,0.0,7
8,2026-02-06 08:00:00,73,0.0,8
9,2026-02-06 09:00:00,73,0.0,9
